# 05 · Validation plan — novelty budget + synthesis plan with paired controls

**Standard slot:** *validation plan.* **For Project 03 this means:** convert the frontier into a
**"novelty budget"** guideline (D★), select a **novel-but-foldable set**, and write a costed
synthesis/expression plan whose controls are a **HIGH-novelty risky design paired with a conservative
design**, plus an unrelated control (D4/D5). Optional: a short MD stability check on top picks.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Derive the novelty budget

From the frontier (nb 04), state — **per topology and length band** — the most novelty (lowest TM to
the PDB) you can spend while keeping scRMSD < 2 Å. That is the budget: a practical rule for "how far
from natural fold space you can push this topology/length before foldability collapses." 

In [ ]:
import pandas as pd, numpy as np
import rfdiff_tools as rt

bb = pd.read_csv("results/backbones.csv")
foldable = bb[bb["scrmsd"] < rt.SELF_CONSISTENT_SCRMSD]

# Novelty budget = lowest TM (most novel) still foldable, per topology (and length band).
budget = (foldable.groupby("ss_bias")
          .agg(min_tm_foldable=("tm_to_pdb", "min"),
               median_tm_foldable=("tm_to_pdb", "median"),
               n_foldable=("tm_to_pdb", "size")).round(3))
print("NOVELTY BUDGET (lower min_tm = more novelty affordable while still foldable):")
print(budget)
print("\nRule of thumb to state in your thesis (fill from YOUR data, not the synthetic mock):")
print("  e.g. 'all-alpha tolerates TM down to ~X at L<=120; all-beta only to ~Y; long backbones less.'")

## 2 · Select the novel-but-foldable set (D★)

Pick the designs that are both foldable (scRMSD < 2 Å) and genuinely novel (TM < 0.5), ranked by a
simple novelty-weighted-foldability score. **On the mock backend these are synthetic `EXAMPLE_DATA`
ids** — replace with your real RFdiffusion designs for the actual synthesis list.

In [ ]:
cand = bb[(bb["scrmsd"] < rt.SELF_CONSISTENT_SCRMSD) & (bb["tm_to_pdb"] < rt.NOVEL_TM)].copy()
# score: reward low scRMSD AND low TM (more novel). Tune weights for your priorities.
cand["pick_score"] = (rt.SELF_CONSISTENT_SCRMSD - cand["scrmsd"]) + (rt.NOVEL_TM - cand["tm_to_pdb"])
selected = cand.sort_values("pick_score", ascending=False).head(8)
selected.to_csv("results/selected_novel_foldable.csv", index=False)
print("Selected novel-but-foldable set (label EXAMPLE_DATA if synthetic):")
print(selected[["backbone_id", "length", "ss_bias", "scrmsd", "plddt", "tm_to_pdb"]].to_string(index=False))

## 3 · The synthesis / expression plan with PAIRED controls

Controls are mandatory and, for this project, **paired by novelty** — that pairing is what tests
whether the novelty budget holds up experimentally.

In [ ]:
plan = """# Synthesis & Expression Plan (Project 03 — by <name>, <date>)

## Designs to synthesize (codon-optimized, gene synthesis via an IGSC-screening provider)
1. TEST    — one HIGH-novelty design (lowest TM, scRMSD<2): the risky case.
2. CONTROL+ — one CONSERVATIVE design (high TM ~>0.6, very low scRMSD, high pLDDT): expected to fold.
3. CONTROL- — an UNRELATED natural protein of similar size (e.g., a small natural monomer): orthogonal control.
   (The risky vs conservative PAIR directly tests the novelty budget.)

## Expression
- Host: E. coli BL21(DE3); T7 vector; His6 tag + TEV site; 16-18 C overnight induction.
- Note: switch to a refolding protocol or mammalian expression only if the novel design is insoluble.

## Characterization (go/no-go -> basic -> deep)
- go/no-go : express -> SDS-PAGE (right MW?) -> SEC (monodisperse monomer?).
- basic    : CD (secondary-structure content vs the DESIGNED topology), DSF (Tm / thermostability).
- deep     : crystallography or cryo-EM to CONFIRM the novel fold; SEC-MALS / SAXS for solution shape.

## Read-out tied to the hypothesis
- Does the HIGH-novelty design fold (SEC monomer + CD matching topology) as well as the conservative one?
- If the conservative folds and the novel does not, the novelty budget was exceeded for that topology/length.

## Controls, timeline, cost
- Controls: positive (conservative design), negative (unrelated protein); both run in parallel.
- Timeline: synthesis ~2-3 wk; expression/purification ~2 wk; characterization ~3-4 wk.
- Cost: gene synthesis (3 constructs) + expression reagents + SEC/CD/DSF time — itemize for your advisor.

## Responsible research
- Low dual-use: novel monomers with no designed function. Synthesis through an IGSC-screening provider;
  institutional biosafety/ethics approval before any wet-lab work. Re-evaluate under MASTER_BLUEPRINT §7
  if a fold is later repurposed toward a functional target.
"""
open("results/synthesis_plan.md", "w").write(plan)
print("wrote results/synthesis_plan.md — fill <name>/<date> and the costed reagent list.")

## 4 · (Stretch) Short MD stability check on top picks `[stretch]`

Run a short MD (OpenMM, ~10–50 ns) on the top picks and check the backbone doesn't drift far from the
designed structure (`md_rmsd`). Feed `md_rmsd` into the shared filter's Layer 4 (`dynamics_filter`)
for an extra confidence layer on the designs you'll actually synthesize. This is a **scaffold** —
implement on a GPU runtime; it does not run on the mock backend.

In [ ]:
# OpenMM scaffold (GPU runtime; small systems feasible on T4, long MD -> HPC). Illustrative only.
#
#   # for a top-pick PDB:
#   from openmm.app import PDBFile, ForceField, Modeller, Simulation, PME, HBonds
#   from openmm import LangevinMiddleIntegrator, unit
#   pdb = PDBFile("results/backbones/<top_pick>.pdb")
#   ff = ForceField("amber14-all.xml", "amber14/tip3pfb.xml")
#   # ... solvate, minimize, equilibrate, run 10-50 ns, compute backbone RMSD vs frame 0 ...
#   md_rmsd = ...  # mean backbone RMSD over the trajectory (A)
#
# then feed into the shared filter's Layer 4:
#   import filtering_pipeline as fp
#   d = fp.Design(design_id="top_pick", sequence="...", design_type="monomer",
#                 scrmsd=..., plddt=..., md_rmsd=md_rmsd)
#   fp.dynamics_filter(d, max_md_rmsd=3.0)   # stable if the fold doesn't drift
print("Stretch task — run OpenMM on top picks (GPU); feed md_rmsd into fp.dynamics_filter (Layer 4).")

## D4 / D5 checklist
- [ ] **Novelty budget** guideline derived from the frontier, per topology/length (D★).
- [ ] `results/selected_novel_foldable.csv`: the novel-but-foldable set (real designs, not EXAMPLE_DATA, for synthesis).
- [ ] `results/synthesis_plan.md`: paired HIGH-novelty vs conservative controls + unrelated control, costed + timed.
- [ ] (Stretch) short MD stability check on top picks; `md_rmsd` fed into Layer 4.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release including the novelty-budget guideline.

You're done — you've mapped the novelty–foldability frontier and turned it into a budget the cohort can use.